# Experimento de Filtro Casado no GNU Radio

## Objetivo

Demonstrar o funcionamento de um **filtro casado** (*matched filter*) utilizando o GNU Radio.

O sistema simula uma transmissão digital simples, na qual:

- bits aleatórios são gerados;
- os bits são transformados em pulsos;
- ruído é adicionado ao canal;
- um filtro casado é aplicado no receptor;
- observa-se a melhora na detectabilidade do sinal.

O experimento ilustra um dos conceitos mais importantes de telecomunicações:

> a detecção eficiente de sinais conhecidos na presença de ruído.

---

# Introdução

Em sistemas de comunicação reais, o receptor raramente recebe um sinal “limpo”.

Durante a transmissão, diversos fenômenos degradam o sinal:

- ruído térmico;
- interferências;
- distorções;
- perdas no canal;
- imperfeições eletrônicas.

Assim, o sinal recebido pode ser modelado como:

$$
r(t)=s(t)+n(t)
$$

onde:

- $s(t)$ é o sinal transmitido;
- $n(t)$ é o ruído;
- $r(t)$ é o sinal recebido.

O grande desafio do receptor é determinar:

> “qual símbolo foi transmitido?”

mesmo quando o sinal está parcialmente escondido no ruído.

---

# O Conceito de Filtro Casado

O filtro casado é um filtro projetado especificamente para detectar um sinal conhecido.

A ideia central é:

> construir um filtro cuja resposta seja compatível com o formato do sinal transmitido.

Quando o sinal correto chega ao receptor:

- sua energia é acumulada de forma coerente;
- o ruído tende a se cancelar parcialmente;
- a relação sinal-ruído aumenta.

---

# Formulação Matemática

Se o sinal transmitido é:

$$
s(t)
$$

o filtro casado possui resposta ao impulso:

$$
h(t)=s(T-t)
$$

ou seja:

- uma versão invertida no tempo do sinal transmitido;
- podendo incluir conjugado complexo em sistemas complexos.

No caso discreto:

$$
h[n]=s[N-1-n]
$$

---


O filtro casado funciona de maneira muito semelhante a uma operação de correlação.

O receptor continuamente pergunta:

> “o sinal recebido se parece com o sinal esperado?”

Quando há grande semelhança:

- ocorre soma construtiva;
- aparece um pico na saída;
- o símbolo pode ser detectado.

---

# Estrutura do Experimento

O fluxo desenvolvido no GNU Radio possui a seguinte estrutura:

```text
Random Source
   ↓
Chunks to Symbols
   ↓
Repeat
   ↓
Add ← Noise Source
   ↓
Throttle
   ├──→ QT GUI Time Sink (Sem filtro)
   ↓
FIR Filter (Filtro Casado)
   ↓
QT GUI Time Sink (Com filtro)

![Texto Alternativo](diagrama.png)

![Texto Alternativo](image.png)

# Descrição

---

# 1. Random Source

## Função

O bloco `Random Source` é responsável por gerar a sequência de bits que será transmitida pelo sistema.

Esses bits representam a informação digital que será enviada pelo transmissor.

Exemplo de sequência gerada:

```text
0 1 1 0 1 0 0 1 ...
```

---

Esse bloco simula uma fonte de informação digital, equivalente a:

- dados de um computador;
- bits de uma transmissão Wi-Fi;
- informações de um sistema de rádio digital;
- pacotes de dados em telecomunicações.

---

Ele fornece o sinal binário original que será convertido em pulsos e posteriormente contaminado pelo ruído.

---

# 2. Chunks to Symbols

## Função

O bloco `Chunks to Symbols` converte os bits gerados em níveis de amplitude.

Foi utilizada a tabela:

```python
[-1, 1]
```

Assim:

```text
0 → -1
1 → +1
```

A sequência digital torna-se:

```text
-1 +1 +1 -1 +1 ...
```

---

Esse processo representa a modulação digital do sinal.

O sistema deixa de trabalhar apenas com bits abstratos e passa a utilizar amplitudes físicas transmissíveis.

Essas amplitudes podem representar:

- tensão elétrica;
- potência de rádio;
- intensidade luminosa;
- amplitude de portadora.

---

# 3. Repeat

## Função

O bloco `Repeat` aumenta a duração temporal de cada símbolo.

Se:

```python
sps = 8
```

então cada símbolo será repetido 8 vezes.

Exemplo:

```text
+1
```

torna-se:

```text
+1 +1 +1 +1 +1 +1 +1 +1
```

---


Esse bloco cria um pulso com duração finita.

Na prática, sistemas de comunicação nunca transmitem bits instantâneos. Em vez disso, transmitem pulsos temporais.

Nesse experimento, o pulso possui formato retangular.

---

## Formação do Pulso

Após o bloco `Repeat`, o sinal passa a possuir estrutura temporal.

O símbolo deixa de ser apenas um valor discreto e passa a ocupar uma janela de tempo.

Isso é fundamental para:

- transmissão física;
- sincronização;
- detecção no receptor.

---

# 4. Noise Source

## Função

O bloco `Noise Source` gera ruído gaussiano aleatório.

Esse ruído é somado ao sinal transmitido para simular imperfeições do canal.

---

Esse ruído representa fenômenos reais presentes em sistemas físicos:

- ruído térmico;
- interferência eletromagnética;
- imperfeições eletrônicas;
- distorções do canal.

---

## Importância no Experimento

Sem ruído, o problema de detecção seria trivial.

O objetivo do filtro casado é justamente recuperar o sinal mesmo quando ele está parcialmente escondido no ruído.

---

# 5. Add

## Função

O bloco `Add` soma:

- o sinal transmitido;
- o ruído gerado.

O sinal recebido torna-se:

$$
r(t)=s(t)+n(t)
$$

onde:

- $s(t)$ representa o sinal transmitido;
- $n(t)$ representa o ruído;
- $r(t)$ representa o sinal recebido.

---

Esse bloco representa o canal de comunicação.

Em sistemas reais, o receptor nunca recebe apenas o sinal puro.

Sempre existe degradação causada pelo meio de transmissão.

---

## Resultado Observado

Após essa etapa:

- o sinal torna-se ruidoso;
- os símbolos tornam-se mais difíceis de identificar;
- a relação sinal-ruído diminui.

---

# 6. Throttle

## Função

O bloco `Throttle` controla a velocidade da simulação.

Ele limita a taxa de processamento de amostras.

---


Como o experimento não utiliza hardware SDR real, o GNU Radio precisa de um mecanismo para impedir que a simulação execute rapidamente demais.

Sem o `Throttle`, o programa tentaria processar os dados na máxima velocidade possível, consumindo excessivamente a CPU.

---

O bloco garante:

- execução estável;
- sincronismo visual;
- funcionamento adequado das interfaces gráficas.

---

# 7. QT GUI Time Sink (Sem Filtro)

## Função

Esse bloco exibe o sinal no domínio do tempo antes da aplicação do filtro casado.

---

## O que é observado?

O sinal apresenta:

- grande quantidade de ruído;
- pulsos parcialmente ocultos;
- baixa clareza visual.

---

Esse gráfico representa exatamente o que um receptor recebe em um sistema real:

```text
sinal desejado + ruído
```

O receptor ainda não consegue distinguir claramente os símbolos transmitidos.

---

# 8. FIR Filter — Filtro Casado

## Função

O bloco `FIR Filter` implementa o filtro casado.

Os coeficientes utilizados são:

```python
[1.0/sps]*sps
```

Para:

```python
sps = 8
```

os taps tornam-se:

$$
h[n]=\left[\frac18,\frac18,\frac18,\frac18,\frac18,\frac18,\frac18,\frac18\right]
$$

---

## Por que isso é um filtro casado?

O pulso transmitido possui formato:

```text
1 1 1 1 1 1 1 1
```

O filtro utiliza exatamente esse mesmo formato.

Assim:

- o sinal desejado soma construtivamente;
- o ruído soma aleatoriamente.

---

## Interpretação Matemática

A saída do filtro é dada por:

$$
y[n]=\sum_k x[k]h[n-k]
$$

Essa operação corresponde à convolução entre o sinal recebido e o filtro.

---

Quando o pulso correto chega:

- todas as amostras possuem o mesmo sinal;
- ocorre acúmulo coerente de energia;
- aparece um pico na saída.

Já o ruído:

- possui comportamento aleatório;
- tende a se cancelar parcialmente.

---

## Consequência

A relação sinal-ruído melhora significativamente.

Isso torna a detecção dos símbolos muito mais robusta.

---

# 9. QT GUI Time Sink (Com Filtro)

## Função

Esse bloco exibe o sinal após a aplicação do filtro casado.

---

Após a filtragem:

- os pulsos tornam-se mais claros;
- o ruído aparenta ser reduzido;
- os níveis dos símbolos ficam mais definidos;
- a detecção visual torna-se muito mais simples.

---

## Interpretação Conceitual

Esse bloco demonstra visualmente o ganho proporcionado pelo filtro casado.

O receptor agora consegue reconhecer a estrutura do sinal transmitido mesmo na presença de ruído.

---

# Aplicações dos Filtros Casados

---

# 1. Comunicações Digitais

Filtros casados são amplamente utilizados em sistemas modernos de comunicação digital.

Seu principal objetivo é melhorar a detecção de símbolos recebidos na presença de ruído.

---

## Aplicações

- Wi-Fi;
- LTE/4G/5G;
- rádio digital;
- comunicação via satélite;
- enlaces ópticos.

---

O filtro casado:

- maximiza a relação sinal-ruído;
- melhora a confiabilidade da detecção;
- reduz a taxa de erro de bits.

---

# 2. Radar

Filtros casados são fundamentais em sistemas de radar.

O radar transmite um pulso conhecido e aguarda o eco refletido pelo alvo.

O receptor utiliza um filtro casado para detectar esse eco no meio do ruído.

---

Quando o eco recebido coincide com o formato esperado:

- ocorre correlação máxima;
- surge um pico na saída;
- o alvo é detectado.

---

- radar aeronáutico;
- radares automotivos;
- radares meteorológicos;
- radares militares.

---

# 3. GPS e Sistemas GNSS

Sistemas como GPS utilizam filtros casados para detectar os códigos pseudoaleatórios transmitidos pelos satélites.

---

## Função

O receptor:

- correlaciona o sinal recebido com códigos conhecidos;
- identifica os satélites visíveis;
- realiza sincronização temporal.

---

Sem técnicas de correlação e matched filtering, a recepção GPS seria extremamente difícil devido ao baixo nível de potência dos sinais recebidos.

---

# 4. Sonar

Filtros casados também são utilizados em sistemas sonar.

---

- detecção de objetos submersos;
- navegação submarina;
- mapeamento oceânico.

---

## Funcionamento

Assim como no radar:

- um pulso acústico é transmitido;
- o eco refletido retorna;
- o filtro casado detecta o eco no meio do ruído.

---

# 5. Processamento de Imagens

Filtros casados podem ser utilizados para reconhecimento de padrões em imagens.

---

- detecção de objetos;
- reconhecimento biométrico;
- processamento médico;
- visão computacional.

---

O sistema procura padrões específicos previamente conhecidos dentro da imagem.

---

# 6. Astronomia e Radioastronomia

Sinais extremamente fracos provenientes do espaço podem ser detectados utilizando técnicas de matched filtering.

---

- detecção de pulsares;
- análise de sinais cósmicos;
- observação de rádio galáxias.

---